In [ ]:
"""Quant Finance: o fator Value com dados reais e bootstrap em blocos.

Versão 2 do projeto Value Factor. Na versão 1 (500 ações simuladas) os retornos
eram paramétricos: normais, com a correlação entre valor e momentum quase zero
por construção. Aqui os retornos vêm de DADOS REAIS: os fatores mensais HML
(valor) e Mom (momentum) da Biblioteca de Dados de Kenneth French.

Os "mundos alternativos" de 30 anos são montados por bootstrap estacionário em
blocos (Politis e Romano): sorteiam-se pedaços de meses reais consecutivos, e a
mesma data vale para valor e momentum. Isso preserva caudas gordas, a correlação
real entre os dois fatores e parte do agrupamento de volatilidade, sem assumir
uma distribuição. O trabalho agora é nos fatores, não nas 500 ações da versão 1.

Limites: o bootstrap trata a história observada como a população. Ele não cria
regimes que não aconteceram, e o prêmio estimado já é o de uma amostra que
pode ter sido publicada e depois enfraquecido. Material educacional.

Dados: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html
Requer: numpy e matplotlib.
"""
import io
import os
import urllib.request
import zipfile

os.makedirs("graficos_value_v2", exist_ok=True)
os.makedirs("dados_french", exist_ok=True)

#Passo 1: parâmetros
import numpy as np

rng = np.random.default_rng(42)
BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
ARQ_FATORES = "F-F_Research_Data_Factors_CSV.zip"   # Mkt-RF, SMB, HML, RF
ARQ_MOM = "F-F_Momentum_Factor_CSV.zip"             # Mom
anos = 30
T = anos * 12            # meses em cada mundo alternativo
n_mundos = 5000
bloco = 12               # tamanho médio do bloco do bootstrap (meses)

#Passo 2: baixar e ler os fatores reais
def baixar(nome):
    destino = os.path.join("dados_french", nome)
    if not os.path.exists(destino):
        try:
            req = urllib.request.Request(BASE + nome, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                conteudo = r.read()
        except Exception as erro:
            raise SystemExit(
                f"Não consegui baixar {nome} ({erro}).\n"
                "Baixe os 2 arquivos CSV.zip em "
                "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html "
                "e salve na pasta dados_french/."
            )
        with open(destino, "wb") as f:
            f.write(conteudo)
    return destino

def ler_french(caminho):
    """Lê só o bloco MENSAL do arquivo (o anual vem depois). Devolve nomes, datas, retornos."""
    with zipfile.ZipFile(caminho) as z:
        nome = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
        linhas = z.read(nome).decode("latin-1").splitlines()
    cab, datas, vals = None, [], []
    for i, ln in enumerate(linhas):
        cel = [c.strip() for c in ln.split(",")]
        if len(cel[0]) == 6 and cel[0].isdigit():          # linha mensal: AAAAMM
            if cab is None:                                 # o cabeçalho está acima
                j = i - 1
                while j >= 0 and not (linhas[j].startswith(",") and len(linhas[j].split(",")) > 1):
                    j -= 1
                cab = [c.strip() for c in linhas[j].split(",")][1:]
            datas.append(int(cel[0]))
            vals.append([float(c) for c in cel[1:]])
        elif datas:                                         # acabou o bloco mensal
            break
    vals = np.array(vals)
    ok = ~(vals <= -99).any(axis=1)                         # -99.99 = dado faltante
    return cab, np.array(datas)[ok], vals[ok] / 100

cab_f, dat_f, val_f = ler_french(baixar(ARQ_FATORES))
cab_m, dat_m, val_m = ler_french(baixar(ARQ_MOM))
comum = np.intersect1d(dat_f, dat_m)
hml = val_f[np.isin(dat_f, comum), cab_f.index("HML")]
mom = val_m[np.isin(dat_m, comum), 0]
datas = comum
ano_dec = datas // 100 + ((datas % 100) - 1) / 12          # data em anos decimais
X = np.c_[hml, mom]                                         # linhas = meses, colunas = [valor, momentum]
N = len(X)
print(f"Amostra: {datas[0]} a {datas[-1]} ({N} meses)")

#Passo 3: o que os dados reais mostram
def anual(x):  return x.mean() * 12
def vol_a(x):  return x.std(ddof=1) * np.sqrt(12)
def estat_t(x): return x.mean() / (x.std(ddof=1) / np.sqrt(len(x)))
def curtose(x):
    z = (x - x.mean()) / x.std()
    return (z**4).mean() - 3
def assim(x):
    z = (x - x.mean()) / x.std()
    return (z**3).mean()

mix = 0.5 * hml + 0.5 * mom
corr_real = np.corrcoef(hml, mom)[0, 1]
print(f"{'':<10}{'Retorno a.a.':>14}{'Vol a.a.':>10}{'Sharpe':>8}{'t':>7}{'Assim.':>8}{'Curtose':>9}{'Pior mês':>10}")
for nome, x in (("Valor", hml), ("Momentum", mom), ("Mistura", mix)):
    print(f"{nome:<10}{anual(x):>14.1%}{vol_a(x):>10.1%}{anual(x) / vol_a(x):>8.2f}"
          f"{estat_t(x):>7.2f}{assim(x):>8.2f}{curtose(x):>9.1f}{x.min():>10.1%}")
print(f"Correlação mensal valor x momentum: {corr_real:.2f}")

print("Por década (retorno anualizado):")
ini = (datas[0] // 100 // 10) * 10
for dec in range(ini, datas[-1] // 100 + 1, 10):
    sel = (datas // 100 >= dec) & (datas // 100 < dec + 10)
    if sel.sum() >= 24:
        print(f"  {dec}s: valor {anual(hml[sel]):>6.1%} | momentum {anual(mom[sel]):>6.1%}")

#Gráfico 1: riqueza acumulada real
import matplotlib.pyplot as plt
plt.style.use("dark_background")
amarelo = "#FFDF00"
azul = "#4FC3F7"

def riqueza(r):
    return 100 * np.cumprod(1 + r)

plt.figure(figsize=(7, 4))
plt.plot(ano_dec, riqueza(hml), color=amarelo, label="Valor (HML)")
plt.plot(ano_dec, riqueza(mom), color=azul, label="Momentum")
plt.plot(ano_dec, riqueza(mix), color="white", lw=2, label="Mistura 50/50")
plt.yscale("log")
plt.ylabel("Riqueza (início = 100)")
plt.legend()
plt.savefig("graficos_value_v2/grafico1_riqueza_real.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 4: e se fossem só 30 anos? (janelas móveis reais)
janela = T
def movel(x):
    c = np.cumsum(np.r_[0, x])
    return (c[janela:] - c[:-janela]) / janela * 12

mv_h, mv_m, mv_x = movel(hml), movel(mom), movel(mix)
ano_fim = ano_dec[janela - 1:]

#Gráfico 2: retorno anualizado em janelas de 30 anos
plt.figure(figsize=(7, 4))
plt.plot(ano_fim, mv_h * 100, color=amarelo, label="Valor")
plt.plot(ano_fim, mv_m * 100, color=azul, label="Momentum")
plt.plot(ano_fim, mv_x * 100, color="white", lw=2, label="Mistura 50/50")
plt.axhline(0, color="gray", ls="--")
plt.xlabel("Ano em que a janela de 30 anos termina")
plt.ylabel("Retorno anualizado (%)")
plt.legend()
plt.savefig("graficos_value_v2/grafico2_janelas_30anos.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 5: caudas gordas (o que a versão 1 não tinha)
print(f"Valor: pior mês {hml.min():.1%}, curtose {curtose(hml):.1f}")
for nome, x in (("Valor", hml), ("Momentum", mom)):
    z = (x - x.mean()) / x.std()
    print(f"  {nome}: meses além de 3 desvios: {(np.abs(z) > 3).sum()} "
          f"(normal esperaria {N * 0.0027:.1f})")

#Gráfico 3: retornos mensais reais contra a normal
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4))
for ax, x, nome, cor in ((a, hml, "Valor", amarelo), (b, mom, "Momentum", azul)):
    ax.hist(x * 100, bins=70, density=True, color=cor, alpha=0.8)
    xs = np.linspace(x.min(), x.max(), 300)
    z = (xs - x.mean()) / x.std()
    ax.plot(xs * 100, np.exp(-z**2 / 2) / (x.std() * 100 * np.sqrt(2 * np.pi)), "w", lw=1.5)
    ax.set_yscale("log")
    ax.set_ylim(1e-4, None)
    ax.set_xlabel(f"{nome}: retorno mensal (%)")
    ax.set_title(f"curtose em excesso = {curtose(x):.1f}")
plt.savefig("graficos_value_v2/grafico3_caudas.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 6: o bootstrap em blocos
def indices_bootstrap(n_obs, n_mund, meses, med_bloco, g):
    """Bootstrap estacionário: blocos de meses consecutivos, tamanho médio med_bloco."""
    idx = np.empty((n_mund, meses), dtype=np.int32)
    idx[:, 0] = g.integers(0, n_obs, n_mund)
    for t in range(1, meses):
        novo = g.random(n_mund) < 1 / med_bloco
        prox = (idx[:, t - 1] + 1) % n_obs
        idx[:, t] = np.where(novo, g.integers(0, n_obs, n_mund), prox)
    return idx

def mundos(X_, metodo, g, n_mund=n_mundos, meses=T):
    """Devolve retornos (n_mund, meses, 2) = [valor, momentum] para cada mundo."""
    if metodo == "normal":                      # a hipótese da versão 1
        return g.multivariate_normal(X_.mean(0), np.cov(X_.T), size=(n_mund, meses))
    return X_[indices_bootstrap(len(X_), n_mund, meses, metodo, g)]   # metodo = tamanho do bloco

#Passo 7: métricas de cada mundo
def metricas(R):
    h, m = R[:, :, 0], R[:, :, 1]
    x = 0.5 * (h + m)
    mt = R.shape[1]
    def t_(v):   return v.mean(1) / (v.std(1, ddof=1) / np.sqrt(mt))
    def sh(v):   return v.mean(1) / v.std(1, ddof=1) * np.sqrt(12)
    def mdd(v):
        w = np.cumprod(1 + v, axis=1)
        return (w / np.maximum.accumulate(w, axis=1) - 1).min(axis=1)
    cor = (((h - h.mean(1, keepdims=True)) * (m - m.mean(1, keepdims=True))).mean(1)
           / (h.std(1) * m.std(1)))
    return dict(t_h=t_(h), t_m=t_(m), sh_h=sh(h), sh_m=sh(m), sh_x=sh(x),
                ret_h=h.mean(1) * 12, corr=cor, mdd_h=mdd(h), mdd_x=mdd(x),
                vol_h=h.std(1), vol_m=m.std(1), vol_x=x.std(1))

metodos = {"Normal (como na v1)": "normal", "Bootstrap iid": 1,
           "Blocos de 12 meses": 12, "Blocos de 60 meses": 60}
res = {nome: metricas(mundos(X, m, np.random.default_rng(7))) for nome, m in metodos.items()}

print(f"\n{n_mundos} mundos de {anos} anos, cada um com a mesma amostra real como base")
print(f"{'Método':<22}{'t médio':>8}{'P(t>2)':>8}{'P(ret<0)':>10}{'Pior 5%':>9}"
      f"{'corr':>7}{'MDD valor':>11}{'MDD mist.':>11}")
for nome, r in res.items():
    print(f"{nome:<22}{r['t_h'].mean():>8.2f}{(r['t_h'] > 2).mean():>8.1%}"
          f"{(r['ret_h'] < 0).mean():>10.1%}{np.percentile(r['ret_h'], 5):>9.1%}"
          f"{r['corr'].mean():>7.2f}{np.median(r['mdd_h']):>11.1%}{np.median(r['mdd_x']):>11.1%}")

#Gráfico 4: a distribuição da estatística t depende do método
cores4 = ["#9AA0A6", "#4FC3F7", amarelo, "white"]
plt.figure(figsize=(7, 4))
bins = np.linspace(-4, 8, 80)
for (nome, r), cor in zip(res.items(), cores4):
    dens, _ = np.histogram(r["t_h"], bins=bins, density=True)
    plt.plot((bins[1:] + bins[:-1]) / 2, dens, color=cor, lw=2, label=nome)
plt.axvline(2, color="gray", ls="--")
plt.xlabel("Estatística t do prêmio de valor em 30 anos")
plt.yticks([])
plt.legend()
plt.savefig("graficos_value_v2/grafico4_t_por_metodo.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 8: valor e momentum nos mundos reais
rb = res["Blocos de 12 meses"]
print(f"\nCorrelação valor x momentum: real {corr_real:.2f} | v1 simulada 0.04 | "
      f"mundos (blocos de 12): média {rb['corr'].mean():.2f}, "
      f"5% a 95% = [{np.percentile(rb['corr'], 5):.2f}, {np.percentile(rb['corr'], 95):.2f}]")
melhor = (rb["sh_x"] > np.maximum(rb["sh_h"], rb["sh_m"])).mean()
menos_vol = (rb["vol_x"] < np.minimum(rb["vol_h"], rb["vol_m"])).mean()
print(f"Mistura 50/50: Sharpe maior que o de valor e o de momentum em {melhor:.1%} dos mundos")
print(f"Mistura 50/50: menos volátil que valor e que momentum em {menos_vol:.1%} dos mundos")

#Gráfico 5: correlação e Sharpe
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4))
a.hist(res["Bootstrap iid"]["corr"], bins=50, color="#4FC3F7", histtype="step", lw=2, density=True, label="iid")
a.hist(rb["corr"], bins=50, color=amarelo, histtype="step", lw=2, density=True, label="blocos de 12")
a.axvline(corr_real, color="white", lw=2, label=f"real = {corr_real:.2f}")
a.axvline(0.04, color="gray", ls="--", label="v1 = 0.04")
a.set_xlabel("Correlação valor x momentum em 30 anos")
a.set_yticks([])
a.legend(loc="center right")
for chave, nome, cor in (("sh_h", "Valor", amarelo), ("sh_m", "Momentum", "#4FC3F7"),
                         ("sh_x", "Mistura 50/50", "white")):
    b.hist(rb[chave], bins=50, histtype="step", lw=2, color=cor, density=True, label=nome)
b.axvline(0, color="gray", ls="--")
b.set_xlabel("Sharpe em 30 anos (blocos de 12)")
b.set_yticks([])
b.legend(loc="upper right")
plt.savefig("graficos_value_v2/grafico5_correlacao_sharpe.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()

#Passo 9: teste com prêmio zero (hipótese nula)
Xn = X - X.mean(0)                          # recentra: o prêmio verdadeiro passa a ser zero
nulo = metricas(mundos(Xn, bloco, np.random.default_rng(11)))
t_obs = estat_t(hml[-T:])                   # t do valor nos últimos 30 anos reais
p = (nulo["t_h"] >= t_obs).mean()
print(f"\nÚltimos 30 anos reais: prêmio de valor {anual(hml[-T:]):.1%} a.a., t = {t_obs:.2f}")
print(f"Se o prêmio verdadeiro fosse zero, t >= {t_obs:.2f} apareceria em {p:.1%} dos mundos")
print(f"Com prêmio zero, P(t > 2) = {(nulo['t_h'] > 2).mean():.1%} "
      f"(a normal padrão daria 2.3%)")
print(f"Amostra inteira: t = {estat_t(hml):.2f}")

#Gráfico 6: o t dos últimos 30 anos contra o mundo sem prêmio
plt.figure(figsize=(7, 4))
plt.hist(nulo["t_h"], bins=60, color=amarelo, alpha=0.8, density=True)
plt.axvline(2, color="gray", ls="--", label="t = 2")
plt.axvline(t_obs, color="#4FC3F7", lw=2.5, label=f"últimos 30 anos reais (t = {t_obs:.2f})")
plt.xlabel("Estatística t com prêmio de valor verdadeiro igual a zero")
plt.yticks([])
plt.legend()
plt.savefig("graficos_value_v2/grafico6_hipotese_nula.png", dpi=200,
            bbox_inches="tight", facecolor="black")
plt.show()